    Instructions:
    1. Run task_0a.py to generate the vector database if not already generated.
    2. Press Run All (or restart kernel and run all cells).
    3. You will be prompted to provide input values.


    – Task 3 (LS1): Implement a program which (a) given one of the feature models, 
    (b) a user specified value of k, (c) one of the four dimensionality reduction 
    techniques (SVD, NNMF, LDA, k-means) chosen by the user, reports the top-k 
    latent semantics extracted under the selected feature space.

    – Store the latent semantics in a properly named output file

    – List imageID-weight pairs, ordered in decreasing order of weights

In [1]:
FEATURE_SPACE = input("Provide a feature space [color, hog, avgpool, layer3, fc, resnet_output].")

DIM_REDUCTION = input("Provide a dimensionality reduction technique [svd, nnmf, lda, kmeans].")

K = int(input("Enter K, the top K latent semantics to extract for the selected feature space."))

In [2]:
from utils.database_utils import retrieve
feature_vectors = retrieve(f'{FEATURE_SPACE}.pt')

print("Generating top-", K, " latent semantics under ", FEATURE_SPACE, " feature space using: ", DIM_REDUCTION)

Generating top- 10  latent semantics under  hog  feature space using:  svd


In [3]:
if DIM_REDUCTION == "svd":
    from feature_reducers.svd import SVDReducer
    reducer = SVDReducer


elif DIM_REDUCTION == "nnmf":
    from feature_reducers.nnmf import NNMFReducer
    reducer = NNMFReducer

elif DIM_REDUCTION == "lda":
    from feature_reducers.lda import LDAReducer
    reducer = LDAReducer

else:
    # kmeans.
    from feature_reducers.kmeans import KMeansReducer
    reducer = KMeansReducer

reducer = reducer(feature_vectors, K)

similarity_matrix = reducer.get_similarity_matrix(feature_vectors)

latent_semantics = reducer.reduce_features(feature_vectors)
print("Top K latent semantics: ")
print(latent_semantics)
print("Shape: ", latent_semantics.shape)

Top K latent semantics: 
[[-6.00082309e+04  6.65768814e+03 -2.76846771e+03 ...  2.51457006e+03
  -2.61168992e+03 -3.09800636e+03]
 [-6.89891671e+04 -7.59311517e+03 -1.51148528e+04 ...  4.57223158e+03
  -5.25013469e+03 -1.47157808e+03]
 [-5.03094595e+04  6.96888529e+03 -7.15917170e+02 ... -3.06543490e+03
  -1.99559876e+03 -3.08267952e+03]
 ...
 [-8.59386905e+04 -1.10725694e+03  3.63386991e+01 ... -5.90086322e+02
  -6.16656774e+03 -1.11381091e+03]
 [-6.97518447e+04 -1.19751610e+04 -8.57533896e+03 ...  1.23244907e+03
   5.81611562e+03 -6.58026581e+02]
 [-6.91182481e+04 -1.65742040e+04 -8.19740960e+03 ... -4.75292639e+03
   8.77406364e+03  2.75365847e+03]]
Shape:  (4339, 10)


In [4]:
# Store the latent semantics in a properly named file.
# We opt to store just the reducer, as we anyway can generate the latent space quickly
# by loading the feature space and passing it to the reducer, eg:
#
# unpicked_reducer = retrieve(f'LS1_color_svd_reducer.pt')
# feature_vectors = retrieve(f'color.pt')
#
# unpickled_reducer.reduce_features(feature_vectors)

from utils.database_utils import store

store(reducer, f'LS1_{FEATURE_SPACE}_{DIM_REDUCTION}_reducer.pt')


 Saving:  LS1_hog_svd_reducer.pt 



In [5]:
# List imageID-weight pairs, ordered in decreasing order of weights

# We are to showcase which images contribute more to each latent feature.
# This is taking the object-feature factor matrix, and sorting by each
# latent feature's weight.

image_weight_tuples = list(zip(feature_vectors.keys(), similarity_matrix))

print("Image ID - weight pairs sorted in descending order of weights for each latent feature:")

for i in range(K):
    print("\n\nLatent feature: ", i + 1)
    for IMG_ID, weight in sorted(image_weight_tuples, key=lambda x : x[1][i], reverse=True):
        print("(ID: ", IMG_ID, ", Weight: ", weight[i], end="),\t")

Image ID - weight pairs sorted in descending order of weights for each latent feature:


Latent feature:  1
(ID:  8616 , Weight:  -0.0020882605814034182),	(ID:  4554 , Weight:  -0.0029388451242582953),	(ID:  3786 , Weight:  -0.0031131759613590823),	(ID:  7428 , Weight:  -0.003142757288311165),	(ID:  3838 , Weight:  -0.003208382299293468),	(ID:  5234 , Weight:  -0.003254781284632115),	(ID:  8668 , Weight:  -0.0034355544953095793),	(ID:  4928 , Weight:  -0.00350603700811408),	(ID:  4664 , Weight:  -0.0035393064564281512),	(ID:  2754 , Weight:  -0.0036032027473214506),	(ID:  7828 , Weight:  -0.004104000718893809),	(ID:  5450 , Weight:  -0.004114899634895555),	(ID:  2806 , Weight:  -0.004189354553120993),	(ID:  5384 , Weight:  -0.004237368843108604),	(ID:  4592 , Weight:  -0.004255784587028019),	(ID:  6516 , Weight:  -0.004488383479228532),	(ID:  4568 , Weight:  -0.004508507017138779),	(ID:  5006 , Weight:  -0.004522180781565831),	(ID:  5102 , Weight:  -0.004548362095801166),	(ID:  3820 , 